In [1]:
%%writefile bitonic_sort.cu
#include <iostream>
#include <vector>
#include <random>
#include <chrono>
#include <cmath>
#include <cuda_runtime.h>
#include <cstdio>

#define CUDA_CHECK(x) \
    do { \
        cudaError_t err = x; \
        if(err != cudaSuccess){ \
            fprintf(stderr, "CUDA Error: %s\n", cudaGetErrorString(err)); \
            exit(1); \
        } \
    } while(0)

bool isSortedAsc(const std::vector<int>& data){
    for(size_t i = 1; i < data.size(); i++){
        if(data[i-1] > data[i]) return false;
    }
    return true;
}

void fillRandom(std::vector<int>& data){
    std::mt19937 rng(std::chrono::high_resolution_clock::now().time_since_epoch().count());
    std::uniform_int_distribution<int> dist(0,1000000000);

    for(auto &x : data)
        x = dist(rng);
}


// CPU
void compare_and_swap(int* arr, int i, int j, int dir){
    bool swap_needed = (arr[i] > arr[j]);
    if(dir == 0) swap_needed = !swap_needed;

    if(swap_needed){
        std::swap(arr[i], arr[j]);
    }
}

void bitonic_merge(int* arr, int low, int size, int dir){
    if(size > 1){
        int k = size/2;
        for(int i = low; i < low+k; i++){
            compare_and_swap(arr, i, i+k, dir);
        }
        bitonic_merge(arr, low, k, dir);
        bitonic_merge(arr, low+k, k, dir);
    }
}

void bitonic_sort_cpu(int* arr, int low, int size, int dir){
    if(size > 1){
        int k = size/2;

        bitonic_sort_cpu(arr, low, k, 1);
        bitonic_sort_cpu(arr, low+k, k, 0);

        bitonic_merge(arr, low, size, dir);
    }
}

void sortCPU(std::vector<int>& data){
    bitonic_sort_cpu(data.data(), 0, data.size(), 1);
}


// GPU
__global__ void bitonic_sort_kernel(int* data, int N, int stage, int step){

    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if(idx >= N) return;

    int k = 1 << stage;
    int j = 1 << step;

    int partner = idx ^ j;   // gets the index to compare with

    if(partner > idx){
        // direction: first half asc, second half desc
        int dir = ((idx & k) == 0);

        if( (data[idx] > data[partner]) == dir ){
            int temp = data[idx];
            data[idx] = data[partner];
            data[partner] = temp;
        }
    }
}


// run CPU and GPU and measure time
void runBenchmark(int N, int runs, double& avg_cpu, double& avg_gpu){

    std::vector<int> cpu(N), gpu(N);

    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));

    int logN = std::log2(N);

    int threads = 256;
    int blocks = (N + threads - 1) / threads;

    int* d_data;
    CUDA_CHECK(cudaMalloc(&d_data, N * sizeof(int)));

    for(int r = 0; r < runs; r++){

        fillRandom(cpu);
        gpu = cpu;

        // CPU time
        auto t0 = std::chrono::high_resolution_clock::now();
        sortCPU(cpu);
        auto t1 = std::chrono::high_resolution_clock::now();
        avg_cpu += std::chrono::duration<double,std::milli>(t1-t0).count();

        // GPU time
        CUDA_CHECK(cudaMemcpy(d_data, gpu.data(), N*sizeof(int), cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaEventRecord(start));

        for(int stage = 1; stage <= logN; stage++){
            for(int step = stage-1; step >= 0; step--){
                bitonic_sort_kernel<<<blocks,threads>>>(d_data, N, stage, step);
            }
        }

        CUDA_CHECK(cudaEventRecord(stop));
        CUDA_CHECK(cudaEventSynchronize(stop));

        float ms = 0;
        CUDA_CHECK(cudaEventElapsedTime(&ms,start,stop));
        avg_gpu += ms;

        CUDA_CHECK(cudaMemcpy(gpu.data(), d_data, N*sizeof(int), cudaMemcpyDeviceToHost));

        if(!isSortedAsc(cpu)) std::cout << "CPU sort failed?\n";
        if(!isSortedAsc(gpu)) std::cout << "GPU sort failed?\n";
    }

    CUDA_CHECK(cudaFree(d_data));
    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));
}



int main(){

    std::vector<int> sizes = {1<<10, 1<<15, 1<<20};
    int runs = 10;

    std::cout << "Bitonic Sort Benchmark\n";
    system("nvidia-smi");

    for(auto N : sizes){

        double cpu = 0, gpu = 0;

        std::cout << "\nSize = " << N << "\n";
        runBenchmark(N, runs, cpu, gpu);

        cpu /= runs;
        gpu /= runs;

        std::cout << "CPU time: " << cpu << " ms\n";
        std::cout << "GPU time: " << gpu << " ms\n";
        std::cout << "Speedup: " << cpu/gpu << "x\n";
    }

    return 0;
}


Writing bitonic_sort.cu


In [2]:
!nvcc bitonic_sort.cu -o bitonic_sort -arch=sm_75

In [3]:
!./bitonic_sort

Bitonic Sort Benchmark
Sun Nov 23 09:00:17 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+------------------------